In [2]:
# Task 1.1
class Combatant:
    def __init__(self, name, health, damage, cooldown):
        self.name = name
        self.health = health
        self.damage = damage
        self.cooldown = cooldown

    def take_damage(self, points):
        self.health -= points

    def is_dead(self):
        return self.health <= 0

In [64]:
# Task 1.2
class Slot:
    def __init__(self, combatant):
        self.combatant = combatant
        self.turns_before_ready = combatant.cooldown
        self.next = None

    def get_priority(self):
        return self.turns_before_ready

    def update_priority(self):
        if self.turns_before_ready > 0:
            self.turns_before_ready -= 1

class PriorityQueue:
    def __init__(self):
        self.head = None
        self.count = 0

    def get_combatants(self):
        result = []

        current = self.head
        while current:
            result.append(current.combatant)
            current = current.next

        return result

    def add(self, combatant):
        combatant = Slot(combatant)
        
        if not self.head:
            self.head = combatant
            self.count += 1
            return

        elif not self.head.next:
            temp = self.head
            
            if combatant.get_priority() < self.head.get_priority():
                self.head = combatant
                self.head.next = temp
            else:
                self.head.next = combatant

            self.count += 1
            return

        previous = self.head
        current = self.head.next
        while current:
            if combatant.get_priority() < current.get_priority():
                previous.next = combatant
                combatant.next = current
                return
            
            previous = current
            current = current.next

        previous.next = combatant

        self.count += 1

    def remove(self, name):
        if not self.head: return

        if self.head.combatant.name == name:
            self.head = self.head.next
            self.count -= 1
            return

        previous = self.head
        current = self.head.next

        while current:
            if current.combatant.name == name:
                previous.next = current.next
                self.count -= 1

            previous = current
            current = current.next

    def pop_highest_priority(self):
        if not self.head: return

        if self.head.get_priority() <= 0:
            temp = self.head
            self.head = self.head.next
            return temp
            
        return

    def update_slots(self):
        if not self.head: return

        current = self.head
        while current:
            current.update_priority()
            current = current.next

In [68]:
# Task 1.3
class Player(Combatant):
    def __init__(self, name, health, damage, cooldown):
        super().__init__(name, health, damage, cooldown)
    
    def get_opponent_choice(self, combatants):
        for i, c in enumerate(combatants):
            print(f"{i+1}. {c.name}")

        index = None
        while not index or not index.isdigit() or int(index) - 1 >= len(combatants):
            print(index)
            index = input('Choose the number of the enemy you want to face: ')

        return combatants[int(index)-1]

class Enemy(Combatant):
    def __init__(self, name, health, damage, cooldown):
        super().__init__(name, health, damage, cooldown)
    
    def get_opponent_choice(self, combatants):
        for c in combatants:
            if isinstance(c, Player):
                return c

In [69]:
# Task 1.4
import csv

with open('./COMBATANTS.txt') as f:
    data = [[int(j) if j.isdigit() else j for j in i] for i in csv.reader(f)][1:]
    print(data)
player = None
queue = PriorityQueue()

for item in data:
    if item[0] == 'player':
        instance = player = Player(*item)
    else:
        instance = Enemy(*item)

    queue.add(instance)

print(queue.get_combatants())

[['player', 50, 8, 3], ['mosquito', 5, 2, 2], ['rat', 10, 5, 3], ['crow', 20, 8, 5]]
[<__main__.Enemy object at 0x7f536009aba0>, <__main__.Player object at 0x7f536009a900>, <__main__.Enemy object at 0x7f5323b0ac10>, <__main__.Enemy object at 0x7f5323b0a850>]


In [70]:
# Task 1.5
while not player.is_dead():
    if len(queue.get_combatants()) <= 1 and queue.get_combatants()[0].name == 'player':
        break
    
    combatant = queue.pop_highest_priority()
    if combatant:
        combatant = combatant.combatant
        chosen = combatant.get_opponent_choice(queue.get_combatants())

        print(f"{combatant.name} dealt {combatant.damage} damage to {chosen.name}")

        chosen.take_damage(combatant.damage)

        if chosen.is_dead():
            print(f"{chosen.name} died!")
            queue.remove(chosen.name)

        queue.add(combatant)

    queue.update_slots()

if player.is_dead():
    print("You died!")
else:
    print("You won!")

mosquito dealt 2 damage to player
1. rat
2. mosquito
3. crow
None


Choose the number of the enemy you want to face:  1


player dealt 8 damage to rat
rat dealt 5 damage to player
mosquito dealt 2 damage to player
crow dealt 8 damage to player
1. rat
2. mosquito
3. crow
None


Choose the number of the enemy you want to face:  3


player dealt 8 damage to crow
rat dealt 5 damage to player
mosquito dealt 2 damage to player
1. crow
2. rat
3. mosquito
None


Choose the number of the enemy you want to face:  3


player dealt 8 damage to mosquito
mosquito died!
crow dealt 8 damage to player
rat dealt 5 damage to player
1. rat
2. crow
None


Choose the number of the enemy you want to face:  2


player dealt 8 damage to crow
rat dealt 5 damage to player
crow dealt 8 damage to player
player died!
You died!
